# Raw to Base

Validate, deduplicate, and write the raw source data to the base layer.

In [ ]:
source_table = "raw.source_data"
target_table = "base.source_data"
dedupe_key = "record_id"

In [ ]:
from pyspark.sql import functions as F

raw_df = spark.table(source_table)
required_columns = {"_ingested_at"}
missing_columns = required_columns.difference(raw_df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

base_df = raw_df.dropDuplicates([dedupe_key]) if dedupe_key in raw_df.columns else raw_df.dropDuplicates()
base_df = base_df.withColumn("_processed_at", F.current_timestamp())
base_df.write.mode("overwrite").format("delta").saveAsTable(target_table)
print(f"Wrote {base_df.count()} rows to {target_table}")